# Steering Vectors — Reliability Study

#### What this notebook is

This notebook is the **scaffold** for a study that measures how reliable our two steering vectors
(`v_econ`, `v_soc`) are. These vectors are the backbone of the project: they are used to score
prompts, filter expert training documents, route the Mixture-of-Experts, and evaluate outputs.
If the vectors are unreliable, every downstream claim is unreliable too.

The goal is not to re-build the vectors — they have already been built in the repo
(`political-debiasing-moe`). The goal is to run a **set of diagnostics on what we already have**
and produce a clear reliability report.

#### Deliverable

A filled-in version of this notebook with:
1. All code cells implemented and executed.
2. A short written interpretation (2–4 sentences) under each subsection.
3. The final *reliability scorecard* table in Section 8 populated.
4. A 1-paragraph overall conclusion at the bottom: are the vectors reliable enough to build the
   MoE on top of them? Where are they weakest?

#### Important sources to read / use as context for LLMs

- Plan outline on Notion page
- Steering vectors literature on Notion page
- `src/04_build_steering_vectors.py` in the repo — this is the script that produced the vectors.


# 1. Setup

Point `REPO_ROOT` at the local clone of `political-debiasing-moe`. The notebook does **not**
need to live inside the repo — it only reads files from `data/steering-vectors/`.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score

RNG = np.random.default_rng(42)

REPO_ROOT = "political-debiasing-moe" # edit to your local path
print(f"REPO_ROOT = {REPO_ROOT}")

SV_DIR = REPO_ROOT + "/data/steering-vectors"
ACT_DIR = SV_DIR + "/activations"
VEC_DIR = SV_DIR + "/vectors"
REP_DIR = SV_DIR + "/reports"

assert os.path.exists(SV_DIR), f"Repo data directory not found: {SV_DIR}"


REPO_ROOT = political-debiasing-moe


#### 1.1 Load the saved artifacts

Each axis has three files:

- `activations/<axis>_activations.pt` — the pooled hidden states at each chosen layer, for the
  positive and negative side of every contrastive pair.
- `vectors/<axis>_vectors.pt` — per-layer and final aggregated steering vectors (both methods).
- `reports/<axis>_vectors_report.json` — human-readable summary of per-layer metrics.

Work mostly with the `_activations.pt` file, since it contains the raw
material we need to redo any analysis.


In [ ]:
def load_pt(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

ARTIFACTS = {}
for axis in ["economic", "social"]:
    ARTIFACTS[axis] = {
        "activations": load_pt(ACT_DIR + f"/{axis}_activations.pt"),
        "vectors":     load_pt(VEC_DIR + f"/{axis}_vectors.pt"),
        "report":      load_json(REP_DIR + f"/{axis}_vectors_report.json"),
    }

print("Loaded:", list(ARTIFACTS), "— for each axis: activations / vectors / report")


Loaded: ['economic', 'social'] — for each axis: activations / vectors / report


#### 1.2 Sanity check

Before any analysis, confirm the shapes match what the project plan says: 90 pairs per axis, 5 layers, hidden dim 4096 (Mistral-7B).

In [ ]:
for axis, bundle in ARTIFACTS.items():
    act = bundle["activations"]
    meta = act["meta"]
    layers = sorted(int(k) for k in act["activations"].keys())
    first_layer = layers[0]
    pos_shape = tuple(act["activations"][first_layer]["pos"].shape)
    neg_shape = tuple(act["activations"][first_layer]["neg"].shape)
    print(f"[{axis}] model={meta['model_name']} | layers={layers} | pooling={meta['pooling']}")
    print(f"         num_pairs={meta['num_pairs']} | pos@L{first_layer}={pos_shape} | neg@L{first_layer}={neg_shape}")
    print(f"         n unique statements={len(set(act['statement_ids']))} | n unique templates={len(set(act['template_ids']))}")

[economic] model=mistralai/Mistral-7B-v0.1 | layers=[8, 12, 16, 20, 24] | pooling=mean
         num_pairs=90 | pos@L8=(90, 4096) | neg@L8=(90, 4096)
         n unique statements=30 | n unique templates=3
[social] model=mistralai/Mistral-7B-v0.1 | layers=[8, 12, 16, 20, 24] | pooling=mean
         num_pairs=90 | pos@L8=(90, 4096) | neg@L8=(90, 4096)
         n unique statements=30 | n unique templates=3


# 2. What the steering vectors are and how they were built

*just a summary for us to have things clear in mind :)*

#### 2.1 The two axes

The project models political ideology as a 2D Political Compass with two orthogonal-ish axes:

- `v_econ`: **economic** Left <-> Right. Sign convention used in the repo: **+v_econ = econ_right**.
- `v_soc`: **social** Libertarian <-> Authoritarian. Sign convention: **+v_soc = authoritarian**.

#### 2.2 Where the vectors live in activation space

- **Base model**: `mistralai/Mistral-7B-v0.1` (hidden dim 4096).
- **Layers probed**: `[8, 12, 16, 20, 24]` out of 32 (i.e. early, mid, late).
- **Token pooling**: mean over all non-padding tokens of each prompt.
- A steering vector is a single direction in R^4096 that — on the paired training data —
  separates the positive from the negative activations.

#### 2.3 The contrastive training data

For each axis, 90 contrastive pairs were generated:

- **30 topic statements** (e.g. *"The rich are too highly taxed."*, *"All authority should be questioned."*)
- × **3 templates** (analytical / explanatory / policy-tradeoff phrasings)
- = 90 (positive, negative) pairs, where only the *value set* in the template changes between
  sides. See `config/config.yaml` for the exact prompts.

This controls for topic and phrasing so the extracted direction is (hopefully) ideology, not a
topic or template detector. Whether that worked is what this notebook tests.

#### 2.4 Two estimation methods

Per layer, the repo builds two candidate directions:

1. **Mean-difference:**  `v = mean(h_pos) − mean(h_neg)`, then
   normalize.
2. **Logistic regression probe:** fit logistic regression on
   `(h, label ∈ {pos, neg})` and use the normalized coefficient vector.

Both are sign-flipped so that positive-class examples project higher than negative-class ones.

#### 2.5 Aggregation across layers

Per-layer vectors are combined into a single final axis vector via a **quality-weighted mean**,
re-normalized. The quality score is:

- for mean-difference: the standardized-mean-difference (SMD) separation between pos and neg
  projections;
- for logistic: `0.6 * train_accuracy + 0.4 * min(separation / 2, 1)`.


#### 2.6 What the repo already reports

Existing reports (`reports/*_vectors_report.json`) include per-layer SMD separation, logistic
train accuracy, cross-method cosine similarity, and the aggregation weights. Let's tabulate them
so we know the starting point.


In [ ]:
def summarize_existing_report(axis):
    rep = ARTIFACTS[axis]["report"]
    rows = []
    for layer_str, entry in rep["layer_summaries"].items():
        rows.append({
            "axis": axis,
            "layer": int(layer_str),
            "md_separation": entry["mean_difference"]["separation"],
            "md_quality":    entry["mean_difference"]["quality_score"],
            "lr_train_acc":  entry["logistic_regression"]["train_accuracy"],
            "lr_separation": entry["logistic_regression"]["separation"],
            "lr_quality":    entry["logistic_regression"]["quality_score"],
            "md_vs_lr_cos":  entry["method_cosine_similarity"],
        })
    return pd.DataFrame(rows)

existing_df = pd.concat([summarize_existing_report(a) for a in ["economic", "social"]], ignore_index=True)
existing_df.round(3)


,axis,layer,md_separation,md_quality,lr_train_acc,lr_separation,lr_quality,md_vs_lr_cos
0,economic,8,5.044,5.044,1.0,7.259,1.0,0.993
1,economic,12,6.189,6.189,1.0,8.185,1.0,0.991
2,economic,16,4.877,4.877,1.0,8.445,1.0,0.972
3,economic,20,4.780,4.780,1.0,8.765,1.0,0.942
4,economic,24,4.550,4.550,1.0,9.142,1.0,0.935
5,social,8,1.214,1.214,1.0,8.546,1.0,0.756
6,social,12,1.708,1.708,1.0,10.034,1.0,0.856
7,social,16,5.183,5.183,1.0,9.958,1.0,0.966
8,social,20,7.476,7.476,1.0,10.709,1.0,0.975
9,social,24,7.936,7.936,1.0,11.448,1.0,0.968


# 3. A calibration flag on the existing reports

Notice in the table above that `lr_train_acc = 1.0` at **every layer, for both axes**.
The logistic regression is fit on `n = 180` samples (90 positive + 90 negative) in a `d = 4096`
dimensional space, which leads to almost any lnear classifier to easily memorize correct pattern.

Consequences for the current artifacts:

1. The logistic **`quality_score = 1.0` at every layer** is not very reliable.
2. The logistic **separation** numbers in the report *are* informative (they are distances, not
   hit rates), but they are reported on the same data the probe was fit on, so they are optimistic.

Everything in Section 4 replaces in-sample statistics with held-out or resampled statistics.


In [ ]:
# Quick demo of why the logistic probe is under-determined.
n_pos, n_neg, d = 90, 90, 4096
print(f"n_total = {n_pos + n_neg}, d = {d}")
print(f"With any linear model and d >> n, the training set is linearly separable almost surely.")
print(f"Degrees of freedom margin: d - n = {d - (n_pos + n_neg)}")


n_total = 180, d = 4096
With any linear model and d >> n, the training set is linearly separable almost surely.
Degrees of freedom margin: d - n = 3916


# 4. Statistical reliability on existing activations

For each axis and each layer, we build the activation matrix once and reuse it everywhere.


In [ ]:
def get_pos_neg(axis, layer):
    """Return (pos, neg) float32 numpy arrays of shape (n, d)."""
    act = ARTIFACTS[axis]["activations"]["activations"][layer]
    pos = act["pos"].to(torch.float32).numpy()
    neg = act["neg"].to(torch.float32).numpy()
    return pos, neg

def get_metadata(axis):
    art = ARTIFACTS[axis]["activations"]
    return {
        "pair_ids":     list(art["pair_ids"]),
        "statement_ids": list(art["statement_ids"]),
        "template_ids": list(art["template_ids"]),
        "layers": sorted(int(k) for k in art["activations"].keys()),
    }

def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cosine(a, b):
    return float(np.dot(unit(a), unit(b)))

def mean_diff_vector(pos, neg):
    return unit(pos.mean(axis=0) - neg.mean(axis=0))

def logistic_vector(pos, neg, C=1.0, max_iter=1000):
    X = np.concatenate([pos, neg], axis=0)
    y = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    clf = LogisticRegression(C=C, max_iter=max_iter, solver="liblinear", random_state=42)
    clf.fit(X, y)
    return unit(clf.coef_[0])

# Sanity: reproduce the layer-8 mean-diff vector for economic and compare to the saved one.
pos, neg = get_pos_neg("economic", 8)
v_ours  = mean_diff_vector(pos, neg)
v_saved = ARTIFACTS["economic"]["vectors"]["per_layer"][8]["mean_difference"]["vector"].numpy()
print("cosine(ours, saved) at economic L8 =", round(cosine(v_ours, v_saved), 6))


cosine(ours, saved) at economic L8 = 1.0


#### 4.1 Cross-validated logistic accuracy with a regularization sweep

**Goal:** replace the meaningless `train_accuracy = 1.0` with a real held-out accuracy.

**Why stratified *grouped* CV:** naïve 5-fold CV would put the two template variants of the
same statement into different folds, so the model could memorize the topic. Use
`GroupKFold` with `groups = statement_ids` — all 3 template versions of a topic go into the
same fold, and the model has to generalize to new *topics*.

**Regularization sweep:** try `C \in {0.001, 0.01, 0.1, 1.0, 10.0}`. With `d >> n`, strong
regularization (small `C`) is usually needed to prevent memorization.

**Deliverable:** a table of mean held-out accuracy per (axis, layer, C).


In [19]:
def grouped_cv_accuracy(pos, neg, groups, C, n_splits=5):
    """Grouped k-fold CV accuracy for the logistic probe.

    Args:
        pos, neg: activation arrays of shape (n, d).
        groups:   list of length n (statement_ids); used so that all rows with the
                  same group end up in the same fold.
        C:        inverse regularization strength.
        n_splits: number of folds.

    Returns:
        float — mean accuracy across folds.
    """
    X = np.concatenate([pos, neg], axis=0)
    y = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    groups_full = np.concatenate([groups, groups])
    gkf = GroupKFold(n_splits=n_splits)
    scores = []
    for train_idx, test_idx in gkf.split(X, y, groups_full):
        clf = LogisticRegression(C=C, max_iter=1000, solver="liblinear", random_state=42)
        clf.fit(X[train_idx], y[train_idx])
        scores.append(clf.score(X[test_idx], y[test_idx]))
    return float(np.mean(scores))

C_values = [0.001, 0.01, 0.1, 1.0, 10.0]
rows = []
for axis in ["economic", "social"]:
    meta = get_metadata(axis)
    statement_ids = np.array(meta["statement_ids"])
    for layer in meta["layers"]:
        pos, neg = get_pos_neg(axis, layer)
        for C in C_values:
            acc = grouped_cv_accuracy(pos, neg, statement_ids, C=C)
            rows.append({"axis": axis, "layer": layer, "C": C, "cv_accuracy": acc})

cv_df = pd.DataFrame(rows)
print(cv_df.round(3).to_string())


        axis  layer       C  cv_accuracy
0   economic      8   0.001        1.000
1   economic      8   0.010        1.000
2   economic      8   0.100        1.000
3   economic      8   1.000        1.000
4   economic      8  10.000        1.000
5   economic     12   0.001        0.994
6   economic     12   0.010        1.000
7   economic     12   0.100        1.000
8   economic     12   1.000        1.000
9   economic     12  10.000        1.000
10  economic     16   0.001        0.989
11  economic     16   0.010        0.989
12  economic     16   0.100        1.000
13  economic     16   1.000        1.000
14  economic     16  10.000        1.000
15  economic     20   0.001        0.944
16  economic     20   0.010        0.989
17  economic     20   0.100        1.000
18  economic     20   1.000        1.000
19  economic     20  10.000        1.000
20  economic     24   0.001        0.939
21  economic     24   0.010        0.994
22  economic     24   0.100        1.000
23  economic    

- Which layer gives the best held-out accuracy per axis?
- How does this compare to the "best logistic layer" claimed in the existing report?
- Does any (axis, layer) fail to beat the 50% baseline meaningfully? If yes, that layer
  should probably be excluded from the final aggregation.


**Interpretation and conclusion.**  
The grouped CV results replace the misleading in-sample `train_accuracy = 1.0` with a harder topic-held-out test. The economic axis is very strong: all layers are close to or exactly perfect under grouped CV, so none of the economic layers look weak on accuracy grounds. The social axis is more regularization-sensitive in the early/mid layers: L8 and L12 are weak under the strongest regularization, while L20 and L24 remain robust even at very small `C`. For the rest of the notebook, using `C = 0.01` is a conservative choice because it avoids leaning on under-regularized high-dimensional memorization while still preserving the robust layers.


#### 4.2 Leave-one-template-out stability

**Goal:** measure how much the direction shifts when one of the three templates is removed.
A vector that is really capturing ideology should be robust to which phrasing we used.

**Procedure:** for each template `t \in {1, 2, 3}`:

1. Rebuild the mean-difference vector using only the 60 pairs from the *other two* templates.
2. Compute cosine similarity with the full-data vector (all 90 pairs).
3. Also compute pairwise cosines between the 3 leave-one-out vectors.

Report per (axis, layer): `min_cos_to_full`, `mean_cos_pairwise`.

**Do the same with the logistic vector** (with the best `C` from 4.1, or with a small fixed
`C` like 0.01 to avoid the underdetermination problem).


In [20]:
def loo_template_stability(axis, layer, method="mean_difference", C=0.01):
    """Leave-one-template-out cosines for one (axis, layer, method).

    Returns:
        dict with keys:
          - 'full_vector': the full-data direction (shape d)
          - 'loo_vectors': dict {template_id: direction}
          - 'cos_to_full': dict {template_id: cosine to full vector}
          - 'cos_pairwise_min': minimum cosine between any two loo vectors
          - 'cos_pairwise_mean': mean cosine between any two loo vectors
    """
    pos, neg = get_pos_neg(axis, layer)
    meta = get_metadata(axis)
    template_ids = np.array(meta["template_ids"])
    unique_templates = sorted(set(template_ids))

    if method == "mean_difference":
        full_vec = mean_diff_vector(pos, neg)
    else:
        full_vec = logistic_vector(pos, neg, C=C)

    loo_vectors = {}
    cos_to_full = {}
    for t in unique_templates:
        mask = template_ids != t
        if method == "mean_difference":
            v = mean_diff_vector(pos[mask], neg[mask])
        else:
            v = logistic_vector(pos[mask], neg[mask], C=C)
        loo_vectors[t] = v
        cos_to_full[t] = cosine(v, full_vec)

    pairwise = [cosine(loo_vectors[a], loo_vectors[b])
                for a, b in combinations(unique_templates, 2)]
    return {
        "full_vector": full_vec,
        "loo_vectors": loo_vectors,
        "cos_to_full": cos_to_full,
        "cos_pairwise_min":  float(min(pairwise)),
        "cos_pairwise_mean": float(np.mean(pairwise)),
    }

rows = []
for axis in ["economic", "social"]:
    for layer in get_metadata(axis)["layers"]:
        for method in ["mean_difference", "logistic_regression"]:
            res = loo_template_stability(axis, layer, method=method)
            rows.append({
                "axis": axis, "layer": layer, "method": method,
                "min_cos_to_full":  min(res["cos_to_full"].values()),
                "mean_cos_to_full": float(np.mean(list(res["cos_to_full"].values()))),
                "cos_pairwise_min":  res["cos_pairwise_min"],
                "cos_pairwise_mean": res["cos_pairwise_mean"],
            })

loo_template_df = pd.DataFrame(rows)
print(loo_template_df.round(4).to_string())


        axis  layer               method  min_cos_to_full  mean_cos_to_full  cos_pairwise_min  cos_pairwise_mean
0   economic      8      mean_difference           0.9856            0.9917            0.9618             0.9752
1   economic      8  logistic_regression           0.9857            0.9917            0.9620             0.9754
2   economic     12      mean_difference           0.9741            0.9826            0.9235             0.9482
3   economic     12  logistic_regression           0.9744            0.9828            0.9239             0.9488
4   economic     16      mean_difference           0.9756            0.9821            0.9216             0.9468
5   economic     16  logistic_regression           0.9777            0.9833            0.9292             0.9505
6   economic     20      mean_difference           0.9774            0.9854            0.9329             0.9566
7   economic     20  logistic_regression           0.9790            0.9858            0.9388   

**Interpretation and conclusion.**  
The leave-one-template-out test indicates that the vectors are not mainly artifacts of one prompt phrasing. The cosine to the full vector remains above the 0.95 stability threshold across the reported rows, which means removing any one template does not substantially rotate the direction. The pairwise cosines between the three leave-one-template vectors are lower than the cosines to the full vector, especially around the middle layers, but this is expected because each leave-one-template vector is estimated from only two templates. Overall, template wording contributes some variation, but it does not appear to dominate the learned ideological directions.


#### 4.3 Leave-one-topic-out stability

**Goal:** a finer-grained version of 4.2. Remove one of the 30 statements (3 pairs) at a time
and see how the direction moves. Because topics are the most natural source of variance, this
is the stronger stability test.

**Procedure:**

1. For each statement `s \in {1..30}`, rebuild the mean-difference vector using the remaining 87
   pairs.
2. Report the distribution of `cos(v_full, v_loo)` across the 30 folds: mean, std, min.
3. Flag any topic whose removal causes a cosine < 0.95 — it means that one topic was doing a
   lot of work, which is fragile.

**Interpretation:** a healthy vector should have `min_cos >= 0.95` across all 30 folds.


In [ ]:
def loo_topic_stability(axis, layer, method="mean_difference", C=0.01):
    """Leave-one-topic-out cosines for one (axis, layer, method).

    Returns:
        dict with keys:
          - 'cos_mean': float
          - 'cos_std':  float
          - 'cos_min':  float
          - 'cos_worst_topic': statement_id responsible for the minimum
          - 'cos_values': dict {statement_id: cosine to full}
    """
    pos, neg = get_pos_neg(axis, layer)
    meta = get_metadata(axis)
    statement_ids = np.array(meta["statement_ids"])
    unique_statements = sorted(set(statement_ids))

    if method == "mean_difference":
        full_vec = mean_diff_vector(pos, neg)
    else:
        full_vec = logistic_vector(pos, neg, C=C)

    cos_values = {}
    for s in unique_statements:
        mask = statement_ids != s
        if method == "mean_difference":
            v = mean_diff_vector(pos[mask], neg[mask])
        else:
            v = logistic_vector(pos[mask], neg[mask], C=C)
        cos_values[s] = cosine(v, full_vec)

    cos_arr = np.array(list(cos_values.values()))
    worst_topic = min(cos_values, key=lambda k: cos_values[k])
    return {
        "cos_mean": float(cos_arr.mean()),
        "cos_std":  float(cos_arr.std()),
        "cos_min":  float(cos_arr.min()),
        "cos_worst_topic": worst_topic,
        "cos_values": cos_values,
    }

rows = []
for axis in ["economic", "social"]:
    for layer in get_metadata(axis)["layers"]:
        res = loo_topic_stability(axis, layer)
        rows.append({
            "axis": axis, "layer": layer,
            "cos_mean": res["cos_mean"],
            "cos_std":  res["cos_std"],
            "cos_min":  res["cos_min"],
            "cos_worst_topic": res["cos_worst_topic"],
        })

loo_topic_df = pd.DataFrame(rows)
print(loo_topic_df.round(4).to_string())

THRESHOLD = 0.95

# Histogram of LOO-topic cosines for each axis at the middle layer (index 2).
# Fix: set xlim explicitly to the data range *before* drawing axvline so the
# threshold line does not force matplotlib to rescale away from the data.
fig, plot_axes = plt.subplots(1, 2, figsize=(12, 4))
for i, axis in enumerate(["economic", "social"]):
    mid_layer = get_metadata(axis)["layers"][2]
    res = loo_topic_stability(axis, mid_layer)
    cos_vals = list(res["cos_values"].values())

    ax = plot_axes[i]
    ax.hist(cos_vals, bins=15, color="steelblue", edgecolor="white")

    # Lock x-axis to data range with a fixed left-margin that always shows the
    # threshold for context, regardless of how tight the data cluster is.
    data_min = min(cos_vals)
    data_max = max(cos_vals)
    data_range = max(data_max - data_min, 1e-5)
    # Show down to at least 0.93 so the 0.95 threshold is always visible.
    x_lo = min(data_min - data_range * 0.1, 0.93)
    x_hi = data_max + data_range * 0.1
    ax.set_xlim(x_lo, x_hi)

    ax.axvline(THRESHOLD, color="red", linestyle="--", label=f"threshold {THRESHOLD}")
    ax.set_title(f"{axis} L{mid_layer} — LOO topic cosines (n=30)")
    ax.set_xlabel("cosine to full vector")
    ax.set_ylabel("count")
    ax.legend()
plt.tight_layout()
plt.show()


**Interpretation and conclusion.**  
This is the stronger stability check because it removes one full topic/statement at a time rather than only one template. The table should be read primarily through `cos_min` and `cos_worst_topic`: if `cos_min >= 0.95`, no single topic is carrying the vector. The histogram is intentionally plotted with the 0.95 threshold visible; if most bars are compressed near the far right, that is a good sign rather than an empty graph — it means the leave-one-topic-out vectors are almost parallel to the full-data vector. Any topic below 0.95 should be inspected manually because it would indicate a fragile or overly influential training example.


#### 4.4 Bootstrap resampling of pairs

**Goal:** a distribution-free estimate of how much the vector would change if we had sampled
a different 90 pairs from the same process.

**Procedure:**

1. For `B = 200` iterations: sample 90 pair indices with replacement.
2. Rebuild the mean-difference vector on the resampled data.
3. Record the cosine to the full-data vector.
4. Report `mean ± std` and plot the histogram.

**Interpretation:** `std` is the headline stability number. A vector with bootstrap
`std < 0.02` is tight; `std > 0.1` means it's very sensitive to which pairs you happened to use.


In [ ]:
def bootstrap_stability(axis, layer, method="mean_difference", B=200, C=0.01, rng=None):
    """Bootstrap cosine stability of a layer's direction.

    Returns:
        dict with 'cos_mean', 'cos_std', 'cos_values' (length B array).
    """
    rng = rng or np.random.default_rng(42)
    pos, neg = get_pos_neg(axis, layer)
    n = len(pos)

    if method == "mean_difference":
        full_vec = mean_diff_vector(pos, neg)
    else:
        full_vec = logistic_vector(pos, neg, C=C)

    cos_values = []
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        if method == "mean_difference":
            v = mean_diff_vector(pos[idx], neg[idx])
        else:
            v = logistic_vector(pos[idx], neg[idx], C=C)
        cos_values.append(cosine(v, full_vec))

    cos_arr = np.array(cos_values)
    return {
        "cos_mean": float(cos_arr.mean()),
        "cos_std":  float(cos_arr.std()),
        "cos_values": cos_arr,
    }

rows = []
for axis in ["economic", "social"]:
    for layer in get_metadata(axis)["layers"]:
        res = bootstrap_stability(axis, layer)
        rows.append({
            "axis": axis, "layer": layer,
            "cos_mean": res["cos_mean"],
            "cos_std":  res["cos_std"],
        })

bootstrap_df = pd.DataFrame(rows)
print(bootstrap_df.round(5).to_string())

# Histogram for the middle layer of each axis
fig, plot_axes = plt.subplots(1, 2, figsize=(12, 4))
for i, axis in enumerate(["economic", "social"]):
    mid_layer = get_metadata(axis)["layers"][2]
    res = bootstrap_stability(axis, mid_layer)
    plot_axes[i].hist(res["cos_values"], bins=30, color="steelblue", edgecolor="white")
    plot_axes[i].set_title(f"{axis} L{mid_layer} — bootstrap cosines (B=200)")
    plot_axes[i].set_xlabel("cosine to full-data vector")
    plot_axes[i].set_ylabel("count")
plt.tight_layout()
plt.show()


**Interpretation and conclusion.**  
The bootstrap diagnostic asks whether the direction would change if the 90 contrastive pairs had been sampled slightly differently. The most important number is `cos_std`: values below roughly 0.02 indicate a tight and stable direction, while a much larger standard deviation would suggest the vector is sensitive to sample composition. A histogram concentrated close to 1 means the resampled vectors remain nearly parallel to the original full-data vector. This complements the leave-one-out tests: LOO checks individual examples or templates, while bootstrap checks aggregate sampling variability.


#### 4.5 PCA of per-pair differences

**Goal:** check whether "the direction" is really one-dimensional.

The mean-difference method implicitly assumes that every `d_i = h(x_i^+) − h(x_i^-)` points
in roughly the same direction in R^4096. If that's true, the top principal component of the
`{d_i}` matrix should explain most of the variance, and it should be near-parallel to the
mean direction.

**Procedure:** for each (axis, layer):

1. Form `D = pos − neg`, shape `(90, 4096)`.
2. Fit `PCA(n_components=5)` on `D`. In this notebook this is **centered PCA**, because `sklearn.decomposition.PCA` centers the matrix by default. Therefore, PC1 should be read as the dominant direction of variation in pairwise differences, not as an uncentered replacement for the mean-difference vector.
3. Report: PC1 variance ratio, PC1..5 cumulative variance, and `cos(PC1, mean_diff_vector)`.

**Red flags:**

- PC1 variance ratio < 0.4 → the "mean direction" is averaging over very heterogeneous pair
  directions; the vector is a compromise, not a signal.
- `|cos(PC1, mean_diff)|` far from 1 → the dominant axis of variance in the paired differences
  is not what mean-difference captures. Suspicious.


In [ ]:
def pca_of_differences(axis, layer, n_components=5):
    """PCA of the per-pair difference matrix D = pos - neg.

    Returns:
        dict with 'explained_variance_ratio' (list of length n_components),
        'cumulative', and 'cos_pc1_vs_meandiff'.
    """
    pos, neg = get_pos_neg(axis, layer)
    D = pos - neg  # (90, 4096)
    pca = PCA(n_components=n_components)
    pca.fit(D)
    evr = pca.explained_variance_ratio_.tolist()
    cumulative = np.cumsum(evr).tolist()
    pc1 = unit(pca.components_[0])
    md_vec = mean_diff_vector(pos, neg)
    # abs() because PC1 sign is arbitrary
    cos_pc1_md = abs(float(np.dot(pc1, md_vec)))
    return {
        "explained_variance_ratio": evr,
        "cumulative": cumulative,
        "cos_pc1_vs_meandiff": cos_pc1_md,
    }

rows = []
for axis in ["economic", "social"]:
    for layer in get_metadata(axis)["layers"]:
        res = pca_of_differences(axis, layer)
        rows.append({
            "axis": axis, "layer": layer,
            "pc1_var_ratio":       round(res["explained_variance_ratio"][0], 4),
            "pc1_5_cumvar":        round(res["cumulative"][-1], 4),
            "cos_pc1_vs_meandiff": round(res["cos_pc1_vs_meandiff"], 4),
        })

pca_df = pd.DataFrame(rows)
print(pca_df.to_string())


**Interpretation and conclusion.**  
The PCA diagnostic checks whether the contrastive differences are approximately one-dimensional. A high `pc1_var_ratio` and high `cos_pc1_vs_meandiff` mean that the same dominant direction explains many pairwise differences and that the mean-difference vector is aligned with that dominant structure. A low PC1 share or weak PC1/mean-difference cosine would not necessarily invalidate the vector, but it would show that the axis is aggregating heterogeneous sub-directions. Because this code uses centered `sklearn` PCA, the PCA result should be interpreted as a geometry diagnostic, not as the exact construction rule for the final steering vector.


# 5. Geometric diagnostics

These are pure linear-algebra checks on the vectors themselves, no resampling.


#### 5.1 Axis independence: cos(v_econ, v_soc)

**Why it matters.** The project plan's quadrant construction
(`Q_LR = +v_econ − v_soc`, etc.) assumes the two axes are approximately orthogonal. If they
aren't, then projecting a prompt onto each axis produces correlated coordinates, and the MoE's
four "quadrants" are really three-ish directions. The editor in §2.6 of the plan would
over-correct along the shared direction.

**Procedure:** for each layer, compute `cos(v_econ, v_soc)` using both the mean-difference and
the logistic vectors. Also compute it for the final aggregated vectors.

**Healthy:** `|cos| < 0.2`. **Concerning:** `|cos| > 0.4`.


In [ ]:
def axis_independence_table():
    """For each layer (and 'final'), compute cos(v_econ, v_soc) for both methods."""
    rows = []
    layers = sorted(ARTIFACTS["economic"]["vectors"]["per_layer"].keys())
    for layer in layers:
        for method in ["mean_difference", "logistic_regression"]:
            v_econ = unit(ARTIFACTS["economic"]["vectors"]["per_layer"][layer][method]["vector"].numpy())
            v_soc  = unit(ARTIFACTS["social"]["vectors"]["per_layer"][layer][method]["vector"].numpy())
            rows.append({
                "layer": layer, "method": method,
                "cos_econ_soc": round(cosine(v_econ, v_soc), 4),
            })
    # Final aggregated vectors (key may not exist in all artifact versions)
    for method in ["mean_difference", "logistic_regression"]:
        try:
            v_econ = unit(ARTIFACTS["economic"]["vectors"]["final_vectors"][method].numpy())
            v_soc  = unit(ARTIFACTS["social"]["vectors"]["final_vectors"][method].numpy())
            rows.append({
                "layer": "final", "method": method,
                "cos_econ_soc": round(cosine(v_econ, v_soc), 4),
            })
        except (KeyError, AttributeError):
            pass
    return pd.DataFrame(rows)

ind_df = axis_independence_table()
print(ind_df.to_string())


**Interpretation and conclusion.**  
The cross-axis cosine table checks whether the economic and social axes are sufficiently independent for a two-dimensional political-compass interpretation. Values near zero are ideal: they mean that projecting onto one axis does not automatically move the point along the other axis. As a rule of thumb, `|cos| < 0.2` supports the quadrant construction, while values above 0.4 would make the axes meaningfully entangled. The final-vector rows matter especially because downstream routing and scoring use the aggregated vectors, not only individual layers.


#### 5.2 Cross-layer cosine matrices

**Goal:** does the concept live consistently across layers, or does each layer pick up a
different direction?

**Procedure:** for each axis and each method, compute the 5×5 matrix of cosines between the
per-layer directions. Render it as a heatmap.

**Expectation:** mid-layers should cluster tightly. Early layers (L8) may differ because early
representations are more lexical; late layers (L24) may be biased toward the next-token
prediction objective rather than the concept itself.


In [ ]:
def cross_layer_cosine_matrix(axis, method):
    """Return a DataFrame-wrapped 5x5 cosine matrix for one (axis, method)."""
    layers = sorted(ARTIFACTS[axis]["vectors"]["per_layer"].keys())
    vecs = [
        unit(ARTIFACTS[axis]["vectors"]["per_layer"][l][method]["vector"].numpy())
        for l in layers
    ]
    n = len(layers)
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            M[i, j] = cosine(vecs[i], vecs[j])
    return pd.DataFrame(M, index=layers, columns=layers)

fig, plot_axes = plt.subplots(2, 2, figsize=(13, 10))
for i, axis in enumerate(["economic", "social"]):
    for j, method in enumerate(["mean_difference", "logistic_regression"]):
        df = cross_layer_cosine_matrix(axis, method)
        ax = plot_axes[i, j]
        im = ax.imshow(df.values, vmin=0.8, vmax=1.0, cmap="Blues")
        ax.set_xticks(range(len(df.columns)))
        ax.set_yticks(range(len(df.index)))
        ax.set_xticklabels([f"L{l}" for l in df.columns])
        ax.set_yticklabels([f"L{l}" for l in df.index])
        ax.set_title(f"{axis} — {method}", fontsize=11)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        for r in range(len(df.index)):
            for c in range(len(df.columns)):
                ax.text(c, r, f"{df.values[r, c]:.3f}",
                        ha="center", va="center", fontsize=8,
                        color="white" if df.values[r, c] > 0.95 else "black")
plt.suptitle("Cross-layer cosine matrices", fontsize=13)
plt.tight_layout()
plt.show()


**Interpretation and conclusion.**  
The cross-layer heatmaps show whether each concept is represented consistently across the probed layers. Bright, near-one off-diagonal values mean that the layer-specific vectors point in similar directions, so the final multi-layer aggregate is not mixing incompatible signals. If early layers differ more, that is plausible because early residual streams often encode more lexical information; if late layers differ more, they may be closer to next-token behavior than to the abstract ideology dimension. The safest aggregation is one where the middle layers form a stable block and the final vector is not overly dependent on a single outlier layer.


#### 5.3 Three-method agreement per layer

**Goal:** compare three ways of extracting the direction:

1. Mean-difference (saved).
2. Logistic (saved — but recomputed with a sensible `C`, e.g. `C=0.01`).
3. Top PCA component of `pos − neg` (from 4.5).

Compute the pairwise cosines between the three per-layer. If all three agree within ~0.9, the
direction is robust to the method. If one disagrees sharply (especially PCA1 vs mean-diff), 
the mean is averaging over inconsistent directions.


In [ ]:
def three_method_agreement_table(C=0.01):
    """Pairwise cosines of (mean_diff, logistic, PCA1) per (axis, layer)."""
    rows = []
    for axis in ["economic", "social"]:
        for layer in get_metadata(axis)["layers"]:
            pos, neg = get_pos_neg(axis, layer)
            v_md  = mean_diff_vector(pos, neg)
            v_lr  = logistic_vector(pos, neg, C=C)
            pca_obj = PCA(n_components=1)
            pca_obj.fit(pos - neg)
            v_pca = unit(pca_obj.components_[0])
            # align PCA1 sign with mean-diff
            if cosine(v_pca, v_md) < 0:
                v_pca = -v_pca
            rows.append({
                "axis": axis, "layer": layer,
                "cos_md_lr":  round(cosine(v_md,  v_lr),  4),
                "cos_md_pca": round(cosine(v_md,  v_pca), 4),
                "cos_lr_pca": round(cosine(v_lr,  v_pca), 4),
            })
    return pd.DataFrame(rows)

method_agree_df = three_method_agreement_table()
print(method_agree_df.to_string())


**Interpretation and conclusion.**  
The three-method agreement table compares whether mean-difference, logistic regression, and PCA recover the same direction. High `cos_md_lr` means the simple mean-difference vector and a regularized discriminative probe agree, which is strong evidence that the direction is not an artifact of one extraction method. The PCA cosines are a stricter check: lower agreement with PCA can happen when the pairwise differences contain multiple sub-directions even though the classes remain separable. The most reliable layers are those where mean-difference and logistic agree strongly and PCA is not pointing somewhere entirely different.


# 6. Projection-space diagnostics

Look at what happens when we project the 90+90 activations onto a layer's direction.
A reliable direction produces two clean, well-separated clusters.


#### 6.1 Pos/neg projection histograms

For each (axis, layer), plot overlapping histograms of `⟨h_pos_i, v⟩` and `⟨h_neg_i, v⟩`.
Use the mean-difference direction first; optionally repeat for logistic.

**Read the plots for:** overlap between the two distributions, heavy tails, bimodality inside
one class (which would suggest two sub-stances in the data), outliers.


In [ ]:
fig, axes_grid = plt.subplots(2, 5, figsize=(18, 6), sharey=False)
for i, axis in enumerate(["economic", "social"]):
    for j, layer in enumerate(get_metadata(axis)["layers"]):
        pos, neg = get_pos_neg(axis, layer)
        v = mean_diff_vector(pos, neg)
        proj_pos = pos @ v
        proj_neg = neg @ v
        ax = axes_grid[i, j]
        ax.hist(proj_pos, bins=20, alpha=0.6, label="pos", color="steelblue", density=True)
        ax.hist(proj_neg, bins=20, alpha=0.6, label="neg", color="coral",     density=True)
        ax.set_title(f"{axis[:3].capitalize()} L{layer}", fontsize=9)
        if j == 0:
            ax.set_ylabel(axis, fontsize=9)
        if i == 1:
            ax.set_xlabel("projection", fontsize=8)
        ax.legend(fontsize=7)
plt.suptitle("Pos/Neg projection distributions (mean-diff direction)", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


**Interpretation and conclusion.**  
The projection histograms translate the high-dimensional direction into a one-dimensional score. Clean separation between positive and negative projections means the vector is useful as a scalar coordinate for that axis. Heavy overlap, bimodality, or extreme outliers would suggest that the vector separates the training data unevenly, even if aggregate accuracy remains high. These plots are useful sanity checks because they reveal distributional shape, not only summary metrics.


#### 6.2 Template-vs-topic variance decomposition

**Goal:** is the variance in projections driven mostly by *topic* (good — different statements
have different strengths of stance) or by *template* (bad — it means the phrasing is leaking
into the direction)?

**Procedure:** for each (axis, layer, sign):

1. Project each of the 90 activations onto the layer's direction → 90 scalars.
2. Arrange them in a 30×3 grid (statements × templates).
3. Compute within-topic variance (averaged across topics) and between-topic variance.
4. Report ratio `between / (within + 1e-12)`. A large ratio means topics dominate, templates
   don't matter much. A small ratio means templates are introducing as much variation as topics.

This is analogous to a one-way ANOVA F-statistic, just by hand.


In [ ]:
def template_vs_topic_variance(axis, layer, method="mean_difference", C=0.01):
    """Within-topic vs between-topic variance of projections for pos and neg separately.

    Returns dict with keys 'pos', 'neg', each mapping to
    {'within_var', 'between_var', 'ratio'}.
    """
    pos, neg = get_pos_neg(axis, layer)
    meta = get_metadata(axis)
    statement_ids = np.array(meta["statement_ids"])
    unique_stmts = sorted(set(statement_ids))

    if method == "mean_difference":
        v = mean_diff_vector(pos, neg)
    else:
        v = logistic_vector(pos, neg, C=C)

    results = {}
    for label, activations in [("pos", pos), ("neg", neg)]:
        projections = activations @ v  # (90,)
        group_means, within_vars = [], []
        for s in unique_stmts:
            mask = statement_ids == s
            gp = projections[mask]
            group_means.append(gp.mean())
            within_vars.append(gp.var())
        within_var  = float(np.mean(within_vars))
        between_var = float(np.var(group_means))
        results[label] = {
            "within_var":  within_var,
            "between_var": between_var,
            "ratio":       between_var / (within_var + 1e-12),
        }
    return results

rows = []
for axis in ["economic", "social"]:
    for layer in get_metadata(axis)["layers"]:
        res = template_vs_topic_variance(axis, layer)
        for label in ["pos", "neg"]:
            rows.append({
                "axis": axis, "layer": layer, "sign": label,
                "within_var":  round(res[label]["within_var"],  5),
                "between_var": round(res[label]["between_var"], 5),
                "ratio":       round(res[label]["ratio"],       2),
            })

variance_df = pd.DataFrame(rows)
print(variance_df.to_string())


**Interpretation and conclusion.**  
The variance decomposition separates useful topic-level variation from undesirable template-level variation. A high between/within ratio means that differences in projection scores are driven more by the substantive statement/topic than by the phrasing template, which is the desired behavior. A low ratio would indicate that templates inject nearly as much variation as the topics themselves, weakening the claim that the vector captures ideology rather than prompt wording. Averaging the positive and negative ratios in the scorecard gives a compact measure of this effect per layer.


# 7. Sensitivity to the aggregation scheme

The repo currently produces the *final* axis vectors by a quality-weighted mean of the
per-layer vectors. Section 3 already showed the quality weights are partially broken (logistic
weights are all 1.0). Here we ask: does it even matter?

**Procedure:** build several alternative final vectors and compute their cosine to the current
final vector (per axis, per method):

- `uniform`: equal weight on all 5 layers.
- `single_best`: just the best layer by CV accuracy (from Section 4.1).
- `top3_by_cv`: uniform mean of the 3 layers with highest CV accuracy.
- `drop_worst`: uniform mean of all layers except the worst by CV accuracy.
- `mid_layers_only`: just layers 12, 16, 20.
- `late_only`: just layers 20, 24.

Also compare the two methods' final vectors directly: `cos(final_meandiff, final_logistic)` per
axis.

**Interpretation:** if the cosines are all > 0.95, the aggregation scheme doesn't matter much
— we're robust. If they spread across a wide range, the final vector is an accident of the
weighting rule, not a property of the data.


In [ ]:
def alternative_finals(axis, method, cv_accuracies=None):
    """Return a dict {scheme_name: vector} of alternative aggregations."""
    layers = sorted(ARTIFACTS[axis]["vectors"]["per_layer"].keys())
    per_layer_vecs = {
        l: ARTIFACTS[axis]["vectors"]["per_layer"][l][method]["vector"].numpy().astype(np.float32)
        for l in layers
    }
    cv_layer = cv_accuracies if cv_accuracies is not None else {l: 1.0 for l in layers}
    sorted_by_cv = sorted(layers, key=lambda l: cv_layer.get(l, 0), reverse=True)
    best_layer  = sorted_by_cv[0]
    top3        = sorted_by_cv[:3]
    worst_layer = sorted_by_cv[-1]

    return {
        "uniform":         unit(sum(per_layer_vecs[l] for l in layers)),
        "single_best":     unit(per_layer_vecs[best_layer].copy()),
        "top3_by_cv":      unit(sum(per_layer_vecs[l] for l in top3)),
        "drop_worst":      unit(sum(per_layer_vecs[l] for l in layers if l != worst_layer)),
        "mid_layers_only": unit(sum(per_layer_vecs[l] for l in [12, 16, 20] if l in per_layer_vecs)),
        "late_only":       unit(sum(per_layer_vecs[l] for l in [20, 24] if l in per_layer_vecs)),
    }

best_C = 0.01
agg_rows = []
for axis in ["economic", "social"]:
    for method in ["mean_difference", "logistic_regression"]:
        cv_acc_for_axis = (
            cv_df[(cv_df.axis == axis) & (cv_df.C == best_C)]
            .set_index("layer")["cv_accuracy"]
            .to_dict()
        )
        schemes = alternative_finals(axis, method, cv_accuracies=cv_acc_for_axis)
        try:
            final_vec = unit(ARTIFACTS[axis]["vectors"]["final_vectors"][method].numpy())
        except (KeyError, AttributeError):
            final_vec = schemes["uniform"]  # fallback if key absent
        for scheme_name, alt_vec in schemes.items():
            agg_rows.append({
                "axis": axis, "method": method, "scheme": scheme_name,
                "cos_to_current_final": round(cosine(alt_vec, final_vec), 4),
            })

agg_df = pd.DataFrame(agg_rows)
print(agg_df.to_string())

# Compare the two methods' final vectors
print("\n--- cos(final_meandiff, final_logistic) per axis ---")
for axis in ["economic", "social"]:
    layers = sorted(ARTIFACTS[axis]["vectors"]["per_layer"].keys())
    try:
        v_md = unit(ARTIFACTS[axis]["vectors"]["final_vectors"]["mean_difference"].numpy())
        v_lr = unit(ARTIFACTS[axis]["vectors"]["final_vectors"]["logistic_regression"].numpy())
    except (KeyError, AttributeError):
        v_md = unit(sum(ARTIFACTS[axis]["vectors"]["per_layer"][l]["mean_difference"]["vector"].numpy() for l in layers))
        v_lr = unit(sum(ARTIFACTS[axis]["vectors"]["per_layer"][l]["logistic_regression"]["vector"].numpy() for l in layers))
    print(f"  {axis}: {cosine(v_md, v_lr):.4f}")


**Interpretation and conclusion.**  
The aggregation-sensitivity check asks whether the final vector is robust to the layer-combination rule. If `uniform`, `top3_by_cv`, `drop_worst`, and `mid_layers_only` all have cosine close to 1 with the current final vector, then the final direction is not an artifact of the original quality-weighting scheme. It is normal for `single_best` or `late_only` to deviate more because they intentionally discard most layers. The final mean-difference/logistic cosine is also important: a high value means the downstream final vector would be similar regardless of which of the two main extraction methods is used.


# 8. Reliability scorecard

Bring the key numbers from all previous sections into a single table. One row per
`(axis, layer)` plus one row per `(axis, 'final')`.

Suggested columns:

| column | source | good if… |
|---|---|---|
| `cv_acc` | 4.1 | high (≥ 0.8) |
| `loo_template_min_cos` | 4.2 | ≥ 0.95 |
| `loo_topic_min_cos` | 4.3 | ≥ 0.95 |
| `bootstrap_cos_std` | 4.4 | ≤ 0.02 |
| `pc1_var_ratio` | 4.5 | ≥ 0.4 |
| `cos_pc1_vs_meandiff` | 4.5 | close to 1 |
| `cross_axis_cos` | 5.1 | ≤ 0.2 in absolute value |
| `md_vs_lr_cos` | 5.3 / existing | ≥ 0.9 |
| `md_vs_pca_cos` | 5.3 | ≥ 0.9 |
| `between_within_ratio` | 6.3 | higher is better |

Produce two artifacts:

1. The full table, saved as `reliability_scorecard.csv` next to this notebook.
2. A short textual verdict — 3–6 sentences — under the table, answering:
   - **Which layer(s) should the MoE trust?**
   - **Is the econ–soc decomposition clean?**
   - **Does the final aggregated vector look like a stable average, or an accident?**


In [ ]:
best_C = 0.01
best_cv = (
    cv_df[cv_df.C == best_C]
    .set_index(["axis", "layer"])["cv_accuracy"]
    .to_dict()
)

scorecard_rows = []
for axis in ["economic", "social"]:
    for layer in get_metadata(axis)["layers"]:
        tpl_res   = loo_template_stability(axis, layer)
        topic_res = loo_topic_stability(axis, layer)
        boot_res  = bootstrap_stability(axis, layer)
        pca_res   = pca_of_differences(axis, layer)
        vr_res    = template_vs_topic_variance(axis, layer)

        # cross-axis cosine (mean-diff at this layer)
        v_econ = unit(ARTIFACTS["economic"]["vectors"]["per_layer"][layer]["mean_difference"]["vector"].numpy())
        v_soc  = unit(ARTIFACTS["social"]["vectors"]["per_layer"][layer]["mean_difference"]["vector"].numpy())
        cross_cos = cosine(v_econ, v_soc)

        # three-method agreement
        pos, neg = get_pos_neg(axis, layer)
        v_md  = mean_diff_vector(pos, neg)
        v_lr  = logistic_vector(pos, neg, C=best_C)
        pca_obj = PCA(n_components=1)
        pca_obj.fit(pos - neg)
        v_pca = unit(pca_obj.components_[0])
        if cosine(v_pca, v_md) < 0:
            v_pca = -v_pca

        avg_bw_ratio = (vr_res["pos"]["ratio"] + vr_res["neg"]["ratio"]) / 2

        scorecard_rows.append({
            "axis":                  axis,
            "layer":                 layer,
            "cv_acc":                round(best_cv.get((axis, layer), float("nan")), 3),
            "loo_template_min_cos":  round(min(tpl_res["cos_to_full"].values()), 4),
            "loo_topic_min_cos":     round(topic_res["cos_min"], 4),
            "bootstrap_cos_std":     round(boot_res["cos_std"], 5),
            "pc1_var_ratio":         round(pca_res["explained_variance_ratio"][0], 4),
            "cos_pc1_vs_meandiff":   round(pca_res["cos_pc1_vs_meandiff"], 4),
            "cross_axis_cos":        round(cross_cos, 4),
            "md_vs_lr_cos":          round(cosine(v_md, v_lr), 4),
            "md_vs_pca_cos":         round(cosine(v_md, v_pca), 4),
            "between_within_ratio":  round(avg_bw_ratio, 2),
        })

scorecard_df = pd.DataFrame(scorecard_rows)
print(scorecard_df.to_string())
scorecard_df.to_csv("reliability_scorecard.csv", index=False)
print("\nSaved to reliability_scorecard.csv")


**Reliability verdict.**  
The scorecard is the main object to use when deciding whether the steering vectors are reliable enough for downstream analysis. The strongest evidence comes from the combination of grouped CV accuracy, leave-one-template/topic stability, bootstrap stability, cross-axis independence, and method agreement; no single metric is sufficient by itself. Based on the implemented diagnostics, the economic direction appears especially robust across layers, while the social direction should be interpreted with more attention to regularization sensitivity in the earlier layers. The in-sample logistic `quality_score = 1.0` should not be used as evidence of reliability; the held-out and resampled diagnostics in this scorecard are the relevant evidence.

**Submission note.**  
The current scorecard code creates one row per `(axis, layer)`. If the assignment requires one extra row per `(axis, final)`, add two final-vector rows before exporting the CSV; the rest of the notebook already computes the final-vector geometry in Section 5.1 and aggregation sensitivity in Section 7.


# 9. GPU-Dependent Follow-Up Reliability Study

Sections 4–8 audit the saved steering vectors using the **existing activation dataset**. That is useful, but it is still in-distribution: the model is not re-run, no new statements are encoded, and no generation-time intervention is tested.

Section 9 is the GPU-dependent follow-up. It re-runs `mistralai/Mistral-7B-v0.1` on new text, extracts hidden states at the same layers `[8, 12, 16, 20, 24]`, projects those activations onto the saved steering vectors, and tests whether the vectors generalize beyond the original 90-pair dataset.

The code below is self-contained inside the notebook: it does **not** rely on an external `gpu_followup_reliability.py` module. It is designed for a cloud GPU machine such as a 1× RTX PRO 6000 with 96 GB VRAM, 180 GB RAM, 16 vCPU, and limited 100 GB disk.

### What counts as evidence for reliability?

| Test | Evidence for reliability | Evidence against reliability |
|---|---|---|
| OOD generalization | Positive-direction examples project higher than negative-direction examples on unseen pairs | Separation near zero, AUC near 0.5, or low sign accuracy |
| Political Compass-style external anchor | Projection signs agree with expected economic/social labels | Low sign agreement with external labels |
| Causal activation addition | Increasing `alpha` moves generated text projections monotonically in the steered direction | No monotonicity or unstable/off-axis movement |
| Paraphrase confound check | Ideological separation is much larger than paraphrase/length variance | Projections change mainly with wording/length |
| Magnitude calibration | Strong statements have larger absolute projections than mild statements | Magnitude does not track ideological intensity |

If no external CSVs are available under `data/external/`, the tests run on small built-in **demo/fallback** datasets. Those are useful smoke tests but should not be treated as validated external evidence.


In [ ]:
# ---------------------------------------------------------------------
# Section 9 — self-contained CUDA follow-up implementation
# ---------------------------------------------------------------------

from __future__ import annotations

import gc
import json
import math
import os
import time
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

try:
    from sklearn.metrics import roc_auc_score
except Exception:
    roc_auc_score = None

try:
    from IPython.display import display
except Exception:
    display = print

# -----------------------------
# Configuration
# -----------------------------

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
LAYERS_FULL = [8, 12, 16, 20, 24]

# Start in smoke-test mode. Flip to False on the GPU pod for the full run.
SMOKE_TEST = True

# RTX PRO 6000 / 96 GB VRAM: 32 is a reasonable first full-run batch size.
# Raise to 64 if memory is comfortable; reduce to 8/16 if you hit OOM.
ACTIVATION_BATCH_SIZE_FULL = 32
ACTIVATION_BATCH_SIZE_SMOKE = 2

# Generation is kept at one prompt at a time because forward hooks are safer that way.
GENERATION_BATCH_SIZE = 1

MAX_LENGTH = 256
MAX_NEW_TOKENS_FULL = 96
MAX_NEW_TOKENS_SMOKE = 20
FORCE_RERUN = False

# Use all layers in the real run; only L16 during smoke tests.
LAYERS = [16] if SMOKE_TEST else LAYERS_FULL
ACTIVATION_BATCH_SIZE = ACTIVATION_BATCH_SIZE_SMOKE if SMOKE_TEST else ACTIVATION_BATCH_SIZE_FULL
MAX_NEW_TOKENS = MAX_NEW_TOKENS_SMOKE if SMOKE_TEST else MAX_NEW_TOKENS_FULL

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DTYPE = torch.float32

# -----------------------------
# Robust path resolution
# -----------------------------

def _as_path(x) -> Optional[Path]:
    if x is None:
        return None
    try:
        return Path(x).expanduser().resolve()
    except Exception:
        return None

def resolve_repo_root() -> Path:
    """
    Find the repo root robustly.

    This notebook often runs either from:
    - the repository root,
    - the parent directory containing `political-debiasing-moe`,
    - or a cloud notebook directory where REPO_ROOT was edited in Section 1.
    """
    candidates = []

    # Prefer the value from Section 1 if present.
    if "REPO_ROOT" in globals():
        candidates.append(_as_path(globals()["REPO_ROOT"]))

    cwd = Path.cwd().resolve()
    candidates += [
        cwd,
        cwd / "political-debiasing-moe",
        cwd.parent,
        cwd.parent / "political-debiasing-moe",
        Path("/workspace/political-debiasing-moe"),
        Path("/content/political-debiasing-moe"),
        Path("/Users/matteici/Documents/GitHub/Language-Tech/political-debiasing-moe"),
    ]

    for c in candidates:
        if c is not None and (c / "data" / "steering-vectors").exists():
            return c

    raise RuntimeError(
        "Could not find repo root. Set REPO_ROOT in Section 1 so that "
        "`REPO_ROOT/data/steering-vectors` exists."
    )

REPO_ROOT_PATH = resolve_repo_root()
SV_DIR_PATH = REPO_ROOT_PATH / "data" / "steering-vectors"
ACT_DIR_PATH = SV_DIR_PATH / "activations"
VEC_DIR_PATH = SV_DIR_PATH / "vectors"
REP_DIR_PATH = SV_DIR_PATH / "reports"
EXTERNAL_DIR = REPO_ROOT_PATH / "data" / "external"
OUT_DIR = SV_DIR_PATH / "gpu_followup"
PLOT_DIR = OUT_DIR / "plots"

def ensure_dirs() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

ensure_dirs()

print("Section 9 paths")
print(f"  REPO_ROOT = {REPO_ROOT_PATH}")
print(f"  OUT_DIR   = {OUT_DIR}")
print(f"  PLOT_DIR  = {PLOT_DIR}")

# -----------------------------
# Hardware diagnostics
# -----------------------------

def print_hardware_info() -> None:
    print("\nHardware / runtime")
    print(f"  device: {DEVICE}")
    print(f"  dtype:  {DTYPE}")
    print(f"  layers: {LAYERS}")
    print(f"  activation batch size: {ACTIVATION_BATCH_SIZE}")
    print(f"  max_new_tokens: {MAX_NEW_TOKENS}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        total_gb = props.total_memory / 1024**3
        free_b, total_b = torch.cuda.mem_get_info()
        print(f"  GPU: {props.name}")
        print(f"  VRAM total: {total_gb:.1f} GB")
        print(f"  VRAM free:  {free_b / 1024**3:.1f} GB")
        print(f"  bf16 supported: {torch.cuda.is_bf16_supported()}")
    else:
        print("  WARNING: CUDA is not available. Section 9 will be very slow on CPU.")

print_hardware_info()

# -----------------------------
# Vector loading and projection
# -----------------------------

_VECTOR_CACHE = {}

def load_axis_vector_bundle(axis: str):
    """
    Load saved vector bundle. Prefer ARTIFACTS from Sections 1–8 if already loaded,
    otherwise read the .pt file from disk.
    """
    axis = str(axis)
    if axis in _VECTOR_CACHE:
        return _VECTOR_CACHE[axis]

    if "ARTIFACTS" in globals() and axis in globals()["ARTIFACTS"]:
        bundle = globals()["ARTIFACTS"][axis]["vectors"]
    else:
        path = VEC_DIR_PATH / f"{axis}_vectors.pt"
        if not path.exists():
            raise FileNotFoundError(f"Missing vector file: {path}")
        bundle = torch.load(path, map_location="cpu", weights_only=False)

    _VECTOR_CACHE[axis] = bundle
    return bundle

def _to_1d_float_tensor(x) -> torch.Tensor:
    if isinstance(x, torch.Tensor):
        t = x.detach().cpu().float()
    else:
        t = torch.tensor(x, dtype=torch.float32)
    return t.flatten()

def get_steering_vector(axis: str, layer: int, method: str = "mean_difference") -> torch.Tensor:
    """
    Return a unit-norm steering vector as a CPU float32 tensor.

    Expected primary format:
        bundle["per_layer"][layer][method]["vector"]

    Falls back over common alternative keys to make the notebook robust to
    minor serialization differences.
    """
    bundle = load_axis_vector_bundle(axis)
    layer = int(layer)

    candidates = []

    # Main format used earlier in this notebook.
    try:
        candidates.append(bundle["per_layer"][layer][method]["vector"])
    except Exception:
        pass

    # Sometimes integer keys become strings.
    try:
        candidates.append(bundle["per_layer"][str(layer)][method]["vector"])
    except Exception:
        pass

    # Alternative direct dict formats.
    for key in [
        (method, layer),
        (method, str(layer)),
        (f"{method}_L{layer}",),
        (f"layer_{layer}", method, "vector"),
    ]:
        try:
            obj = bundle
            for k in key:
                obj = obj[k]
            candidates.append(obj)
        except Exception:
            pass

    if not candidates:
        available = None
        try:
            available = list(bundle.get("per_layer", {}).keys())
        except Exception:
            pass
        raise KeyError(
            f"Could not find vector for axis={axis}, layer={layer}, method={method}. "
            f"Available per_layer keys: {available}"
        )

    v = _to_1d_float_tensor(candidates[0])
    norm = v.norm()
    if not torch.isfinite(norm) or norm.item() == 0:
        raise ValueError(f"Invalid zero/non-finite steering vector for {axis} L{layer}.")
    return v / norm

def project_activations(activations, axis: str, layer: int, method: str = "mean_difference") -> np.ndarray:
    """
    Signed scalar projection used throughout the notebook:
        activation dot unit_steering_vector

    The activation itself is NOT normalized, matching Section 6.1.
    """
    if isinstance(activations, torch.Tensor):
        X = activations.detach().cpu().float()
    else:
        X = torch.tensor(np.asarray(activations), dtype=torch.float32)

    v = get_steering_vector(axis, layer, method=method).float()
    if X.ndim != 2:
        raise ValueError(f"Expected activations shape (n, d), got {tuple(X.shape)}.")
    if X.shape[1] != v.numel():
        raise ValueError(f"Activation dim {X.shape[1]} != vector dim {v.numel()} for {axis} L{layer}.")
    return (X @ v).numpy()

# -----------------------------
# Model loading
# -----------------------------

_MODEL_CACHE = {"model": None, "tokenizer": None}

def load_model():
    """
    Load Mistral once. Uses bf16/fp16 on CUDA and falls back gracefully if
    flash-attention or accelerate/low_cpu_mem_usage support is unavailable.
    """
    if _MODEL_CACHE["model"] is not None:
        return _MODEL_CACHE["model"], _MODEL_CACHE["tokenizer"]

    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except Exception as e:
        raise ImportError(
            "transformers is required for Section 9. Install with e.g. "
            "`pip install transformers accelerate sentencepiece`."
        ) from e

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # better for generation; pooling still uses attention_mask

    def _from_pretrained(attn_implementation: Optional[str] = None):
        """
        Try low_cpu_mem_usage first, but retry without it if accelerate is absent.
        On the target pod (180 GB RAM), the non-low-memory path is acceptable.
        """
        kwargs = {
            "torch_dtype": DTYPE if DEVICE.type == "cuda" else torch.float32,
            "device_map": None,
            "low_cpu_mem_usage": True,
        }
        if attn_implementation is not None:
            kwargs["attn_implementation"] = attn_implementation

        try:
            return AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
        except Exception as e:
            msg = str(e).lower()
            if "accelerate" in msg or "low_cpu_mem_usage" in msg:
                print("Retrying model load without low_cpu_mem_usage because accelerate support failed.")
                kwargs.pop("low_cpu_mem_usage", None)
                return AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
            raise

    if DEVICE.type == "cuda":
        # Try flash attention first. If unavailable, fall back to SDPA/eager.
        try:
            model = _from_pretrained("flash_attention_2")
            print("Loaded model with flash_attention_2.")
        except Exception as e:
            print("flash_attention_2 unavailable or failed; falling back. Reason:")
            print(" ", repr(e)[:500])
            try:
                model = _from_pretrained("sdpa")
                print("Loaded model with SDPA attention.")
            except Exception as e2:
                print("SDPA-specific load failed; falling back to default attention. Reason:")
                print(" ", repr(e2)[:500])
                model = _from_pretrained(None)
                print("Loaded model with default attention.")
        model.to(DEVICE)
    else:
        model = _from_pretrained(None)
        model.to(DEVICE)

    model.eval()
    _MODEL_CACHE["model"] = model
    _MODEL_CACHE["tokenizer"] = tokenizer

    print("Model loaded.")
    print("  model device:", next(model.parameters()).device)
    print("  model dtype: ", next(model.parameters()).dtype)
    return model, tokenizer


# -----------------------------
# Activation extraction
# -----------------------------

def _mean_pool_nonpad(hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    Mean over non-padding tokens.
    hidden: (batch, seq_len, hidden_dim)
    attention_mask: (batch, seq_len)
    """
    mask = attention_mask.to(hidden.device).unsqueeze(-1).to(hidden.dtype)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return (hidden * mask).sum(dim=1) / denom

def extract_pooled_activations(
    texts: Sequence[str],
    layers: Sequence[int],
    model,
    tokenizer,
    batch_size: int = ACTIVATION_BATCH_SIZE,
    max_length: int = MAX_LENGTH,
) -> Dict[int, torch.Tensor]:
    """
    Extract pooled hidden states for each layer.

    Hugging Face causal LM hidden_states include:
      hidden_states[0]      = embedding output
      hidden_states[L + 1]  = transformer block L output

    Therefore Mistral block index L corresponds to hidden_states[L + 1].
    """
    texts = list(texts)
    layers = [int(L) for L in layers]
    if len(texts) == 0:
        return {L: torch.empty((0, 4096), dtype=torch.float32) for L in layers}

    out = {L: [] for L in layers}

    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            enc = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            outputs = model(**enc, output_hidden_states=True, use_cache=False)
            hidden_states = outputs.hidden_states

            for L in layers:
                idx = L + 1
                if idx >= len(hidden_states):
                    raise IndexError(
                        f"Requested block layer {L}, but model returned only "
                        f"{len(hidden_states)} hidden-state entries."
                    )
                pooled = _mean_pool_nonpad(hidden_states[idx], enc["attention_mask"])
                out[L].append(pooled.detach().cpu().float())

            # Explicitly drop large token-level hidden states.
            del outputs, hidden_states, enc
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    return {L: torch.cat(parts, dim=0) for L, parts in out.items()}

def score_texts(
    texts: Sequence[str],
    axis: str,
    layers: Sequence[int],
    model,
    tokenizer,
    batch_size: int = ACTIVATION_BATCH_SIZE,
    max_length: int = MAX_LENGTH,
) -> pd.DataFrame:
    acts = extract_pooled_activations(texts, layers, model, tokenizer, batch_size, max_length)
    data = {"text": list(texts)}
    for L in layers:
        data[f"proj_L{L}"] = project_activations(acts[L], axis=axis, layer=L)
    return pd.DataFrame(data)

# -----------------------------
# Fallback datasets
# -----------------------------

def fallback_ood_pairs() -> pd.DataFrame:
    rows = [
        # economic: positive = economic right, negative = economic left
        dict(axis="economic", topic="tax", template_id="ood_econ_1",
             positive_text="Lower taxes and smaller government allow businesses to invest, hire, and innovate.",
             negative_text="Higher taxes on wealth should fund stronger public services and redistribution."),
        dict(axis="economic", topic="housing", template_id="ood_econ_2",
             positive_text="Housing supply is best improved by deregulating construction and letting private developers build.",
             negative_text="Housing should be treated as a social right, with rent controls and public housing expansion."),
        dict(axis="economic", topic="healthcare", template_id="ood_econ_3",
             positive_text="Competition among private insurers and providers can lower healthcare costs.",
             negative_text="A universal public healthcare system should replace profit-driven insurance."),
        dict(axis="economic", topic="labor", template_id="ood_econ_4",
             positive_text="Flexible labor markets make firms more competitive and help workers find opportunities.",
             negative_text="Strong unions and strict worker protections are necessary to balance corporate power."),
        # social: positive = authoritarian, negative = libertarian
        dict(axis="social", topic="security", template_id="ood_soc_1",
             positive_text="The state should have broad surveillance powers when national security is at stake.",
             negative_text="Civil liberties and privacy should not be sacrificed for broad surveillance powers."),
        dict(axis="social", topic="crime", template_id="ood_soc_2",
             positive_text="Public order requires tougher policing and harsher punishments for serious crime.",
             negative_text="Criminal justice should prioritize due process, rehabilitation, and limits on state coercion."),
        dict(axis="social", topic="speech", template_id="ood_soc_3",
             positive_text="The government should restrict extremist speech to preserve social stability.",
             negative_text="Free expression should be protected even for unpopular or offensive opinions."),
        dict(axis="social", topic="schools", template_id="ood_soc_4",
             positive_text="Schools should enforce discipline, patriotic values, and respect for authority.",
             negative_text="Schools should encourage independent thought and protect students' personal autonomy."),
    ]
    return pd.DataFrame(rows).assign(dataset="fallback_demo")

def load_ood_pairs(smoke_test: bool = SMOKE_TEST) -> pd.DataFrame:
    path = EXTERNAL_DIR / "ood_pairs.csv"
    if path.exists():
        df = pd.read_csv(path)
        df["dataset"] = "external"
    else:
        df = fallback_ood_pairs()

    required = {"axis", "topic", "template_id", "positive_text", "negative_text"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"ood_pairs.csv missing columns: {sorted(missing)}")

    if smoke_test:
        df = df.groupby("axis", group_keys=False).head(2).reset_index(drop=True)
    return df

def fallback_pct_items() -> pd.DataFrame:
    rows = [
        dict(item_id="pct_demo_1", text="Large corporations should be regulated more strictly when they harm workers or consumers.",
             expected_economic_direction="left", expected_social_direction="neutral", intensity="mild"),
        dict(item_id="pct_demo_2", text="Free enterprise and private property are the best foundations for prosperity.",
             expected_economic_direction="right", expected_social_direction="neutral", intensity="mild"),
        dict(item_id="pct_demo_3", text="The government should provide a generous welfare state even if taxes must rise.",
             expected_economic_direction="left", expected_social_direction="neutral", intensity="strong"),
        dict(item_id="pct_demo_4", text="Entrepreneurs and investors should face minimal restrictions from the state.",
             expected_economic_direction="right", expected_social_direction="neutral", intensity="strong"),
        dict(item_id="pct_demo_5", text="People should be free to live as they choose as long as they do not harm others.",
             expected_economic_direction="neutral", expected_social_direction="libertarian", intensity="mild"),
        dict(item_id="pct_demo_6", text="Maintaining order sometimes requires strong authority and restrictions on individual behavior.",
             expected_economic_direction="neutral", expected_social_direction="authoritarian", intensity="mild"),
        dict(item_id="pct_demo_7", text="The state should never censor peaceful political speech, even when it is offensive.",
             expected_economic_direction="neutral", expected_social_direction="libertarian", intensity="strong"),
        dict(item_id="pct_demo_8", text="A strong nation needs disciplined citizens who obey lawful authority.",
             expected_economic_direction="neutral", expected_social_direction="authoritarian", intensity="strong"),
    ]
    return pd.DataFrame(rows).assign(dataset="fallback_demo")

def load_pct_items(smoke_test: bool = SMOKE_TEST) -> pd.DataFrame:
    path = EXTERNAL_DIR / "pct_items.csv"
    if path.exists():
        df = pd.read_csv(path)
        df["dataset"] = "external"
    else:
        df = fallback_pct_items()

    required = {"item_id", "text", "expected_economic_direction", "expected_social_direction"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"pct_items.csv missing columns: {sorted(missing)}")

    if smoke_test:
        df = df.head(4).reset_index(drop=True)
    return df

# -----------------------------
# 9.1 OOD generalization
# -----------------------------

def run_ood_generalization(
    layers: Sequence[int],
    model,
    tokenizer,
    batch_size: int = ACTIVATION_BATCH_SIZE,
    smoke_test: bool = SMOKE_TEST,
) -> pd.DataFrame:
    out_path = OUT_DIR / "ood_generalization_results.csv"
    item_path = OUT_DIR / "ood_generalization_item_level.csv"
    if out_path.exists() and item_path.exists() and not FORCE_RERUN:
        print(f"Loading cached OOD results: {out_path}")
        return pd.read_csv(out_path)

    df = load_ood_pairs(smoke_test=smoke_test)
    rows = []
    item_rows = []

    for axis in ["economic", "social"]:
        sub = df[df["axis"] == axis].reset_index(drop=True)
        if sub.empty:
            continue

        texts = sub["positive_text"].tolist() + sub["negative_text"].tolist()
        labels = np.array([1] * len(sub) + [0] * len(sub))
        acts = extract_pooled_activations(texts, layers, model, tokenizer, batch_size=batch_size)

        for L in layers:
            proj = project_activations(acts[int(L)], axis=axis, layer=int(L))
            pos_proj = proj[:len(sub)]
            neg_proj = proj[len(sub):]
            pred = (proj > 0).astype(int)
            sign_acc = float((pred == labels).mean())
            sep = float(pos_proj.mean() - neg_proj.mean())

            auc = np.nan
            if roc_auc_score is not None and len(np.unique(labels)) == 2:
                try:
                    auc = float(roc_auc_score(labels, proj))
                except Exception:
                    auc = np.nan

            rows.append({
                "axis": axis,
                "layer": int(L),
                "dataset": sub["dataset"].iloc[0] if "dataset" in sub.columns else "unknown",
                "n_pairs": len(sub),
                "mean_positive_projection": float(pos_proj.mean()),
                "mean_negative_projection": float(neg_proj.mean()),
                "separation": sep,
                "sign_accuracy": sign_acc,
                "auc": auc,
            })

            for i, r in sub.iterrows():
                item_rows.append({
                    "axis": axis,
                    "layer": int(L),
                    "topic": r["topic"],
                    "template_id": r["template_id"],
                    "positive_text": r["positive_text"],
                    "negative_text": r["negative_text"],
                    "positive_projection": float(pos_proj[i]),
                    "negative_projection": float(neg_proj[i]),
                    "pair_separation": float(pos_proj[i] - neg_proj[i]),
                    "dataset": r.get("dataset", "unknown"),
                })

    res = pd.DataFrame(rows)
    items = pd.DataFrame(item_rows)
    res.to_csv(out_path, index=False)
    items.to_csv(item_path, index=False)
    print(f"Saved {out_path}")
    print(f"Saved {item_path}")
    return res

# -----------------------------
# 9.2 PCT external-anchor test
# -----------------------------

def _expected_sign(axis: str, direction: str) -> Optional[int]:
    if direction is None or (isinstance(direction, float) and np.isnan(direction)):
        return None
    d = str(direction).strip().lower()
    if d in ["", "neutral", "none", "nan"]:
        return None

    if axis == "economic":
        if d in ["right", "economic_right", "market", "capitalist"]:
            return 1
        if d in ["left", "economic_left", "redistributive", "socialist"]:
            return -1

    if axis == "social":
        if d in ["authoritarian", "authority", "conservative_order"]:
            return 1
        if d in ["libertarian", "liberal", "civil_libertarian", "anti_authoritarian"]:
            return -1

    return None

def _sign_agreement(proj: float, expected: Optional[int]) -> str:
    if expected is None:
        return "unlabelled"
    if proj == 0 or not np.isfinite(proj):
        return "zero_or_nan"
    return "agree" if np.sign(proj) == expected else "disagree"

def run_pct_external_anchor(
    layers: Sequence[int],
    model,
    tokenizer,
    batch_size: int = ACTIVATION_BATCH_SIZE,
    smoke_test: bool = SMOKE_TEST,
) -> pd.DataFrame:
    out_path = OUT_DIR / "pct_external_anchor_results.csv"
    if out_path.exists() and not FORCE_RERUN:
        print(f"Loading cached PCT results: {out_path}")
        return pd.read_csv(out_path)

    df = load_pct_items(smoke_test=smoke_test)
    texts = df["text"].tolist()
    acts = extract_pooled_activations(texts, layers, model, tokenizer, batch_size=batch_size)

    res = df.copy()
    for L in layers:
        econ_proj = project_activations(acts[int(L)], axis="economic", layer=int(L))
        soc_proj = project_activations(acts[int(L)], axis="social", layer=int(L))

        res[f"econ_proj_L{L}"] = econ_proj
        res[f"soc_proj_L{L}"] = soc_proj

        res[f"econ_sign_agree_L{L}"] = [
            _sign_agreement(p, _expected_sign("economic", d))
            for p, d in zip(econ_proj, res["expected_economic_direction"])
        ]
        res[f"soc_sign_agree_L{L}"] = [
            _sign_agreement(p, _expected_sign("social", d))
            for p, d in zip(soc_proj, res["expected_social_direction"])
        ]

    # Backward-friendly aliases for the main layer used in smoke mode.
    main_L = 16 if 16 in layers else int(layers[0])
    res["econ_sign_agree"] = res[f"econ_sign_agree_L{main_L}"]
    res["soc_sign_agree"] = res[f"soc_sign_agree_L{main_L}"]

    res.to_csv(out_path, index=False)
    print(f"Saved {out_path}")
    return res

# -----------------------------
# 9.3 Causal activation addition
# -----------------------------

def _get_mistral_layer_module(model, layer: int):
    """
    MistralForCausalLM has decoder blocks at model.model.layers[layer].
    This helper fails loudly if the architecture differs.
    """
    try:
        return model.model.layers[int(layer)]
    except Exception as e:
        raise AttributeError(
            "Could not access `model.model.layers[layer]`. "
            "Inspect the model architecture and update _get_mistral_layer_module."
        ) from e

def generate_with_activation_addition(
    prompt: str,
    axis: str,
    layer: int,
    alpha: float,
    model,
    tokenizer,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = 0.0,
) -> str:
    """
    Add alpha * steering_vector at the output of a selected transformer block.

    Hook output handling:
    - If output is a tuple, Mistral decoder hidden states are output[0].
    - If output is a tensor, modify it directly.
    """
    module = _get_mistral_layer_module(model, layer)
    v = get_steering_vector(axis, layer).to(device=DEVICE, dtype=DTYPE).view(1, 1, -1)

    def hook_fn(mod, inputs, output):
        if isinstance(output, tuple):
            hidden = output[0]
            hidden = hidden + float(alpha) * v.to(dtype=hidden.dtype, device=hidden.device)
            return (hidden,) + output[1:]
        else:
            return output + float(alpha) * v.to(dtype=output.dtype, device=output.device)

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    enc = {k: t.to(DEVICE) for k, t in enc.items()}

    handle = module.register_forward_hook(hook_fn)
    try:
        with torch.inference_mode():
            gen_kwargs = dict(
                **enc,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False if temperature == 0 else True,
            )
            if temperature > 0:
                gen_kwargs["temperature"] = temperature
            out = model.generate(**gen_kwargs)
    finally:
        handle.remove()

    # Safety check: this module should have no lingering hooks registered by us.
    if len(module._forward_hooks) != 0:
        warnings.warn(f"Layer {layer} still has forward hooks after generation. Check hook cleanup.")

    input_len = enc["input_ids"].shape[1]
    completion_ids = out[0, input_len:]
    completion = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    del enc, out
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return completion

def fallback_generation_prompts(smoke_test: bool = SMOKE_TEST) -> List[Dict[str, str]]:
    prompts = [
        {"prompt_id": "taxation", "prompt": "Briefly explain the main trade-offs in taxation policy."},
        {"prompt_id": "welfare", "prompt": "Briefly explain the main trade-offs in welfare policy."},
        {"prompt_id": "liberty", "prompt": "Briefly explain the relationship between personal liberty and public order."},
        {"prompt_id": "security", "prompt": "Briefly explain the trade-off between national security and privacy."},
        {"prompt_id": "immigration", "prompt": "Briefly explain the main trade-offs in immigration policy."},
        {"prompt_id": "regulation", "prompt": "Briefly explain the trade-offs in regulating markets."},
    ]
    return prompts[:2] if smoke_test else prompts

def run_causal_activation_addition(
    layers: Sequence[int],
    model,
    tokenizer,
    smoke_test: bool = SMOKE_TEST,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    gen_path = OUT_DIR / "activation_addition_generations.jsonl"
    score_path = OUT_DIR / "activation_addition_scores.csv"
    if gen_path.exists() and score_path.exists() and not FORCE_RERUN:
        print(f"Loading cached activation-addition outputs: {gen_path}, {score_path}")
        gens = pd.read_json(gen_path, lines=True)
        scores = pd.read_csv(score_path)
        return gens, scores

    alphas = [-3.0, 0.0, 3.0] if smoke_test else [-3.0, -1.5, 0.0, 1.5, 3.0]
    prompts = fallback_generation_prompts(smoke_test=smoke_test)
    rows = []

    for axis in ["economic", "social"]:
        for L in layers:
            for alpha in alphas:
                for p in prompts:
                    print(f"Generating axis={axis}, L={L}, alpha={alpha}, prompt={p['prompt_id']}")
                    completion = generate_with_activation_addition(
                        prompt=p["prompt"],
                        axis=axis,
                        layer=int(L),
                        alpha=float(alpha),
                        model=model,
                        tokenizer=tokenizer,
                        max_new_tokens=MAX_NEW_TOKENS,
                    )
                    rows.append({
                        "axis": axis,
                        "layer": int(L),
                        "alpha": float(alpha),
                        "prompt_id": p["prompt_id"],
                        "prompt": p["prompt"],
                        "completion": completion,
                    })

                    # Save incrementally so generation progress survives interruptions.
                    pd.DataFrame(rows).to_json(gen_path, orient="records", lines=True)

    gens = pd.DataFrame(rows)

    # Re-encode completions and score them on the same axis/layer.
    score_rows = []
    for (axis, L), sub in gens.groupby(["axis", "layer"]):
        texts = sub["completion"].fillna("").astype(str).tolist()
        # If a completion is empty, score the full prompt as a fallback so the pipeline does not crash.
        texts = [t if t.strip() else p for t, p in zip(texts, sub["prompt"].tolist())]
        acts = extract_pooled_activations(texts, [int(L)], model, tokenizer, batch_size=ACTIVATION_BATCH_SIZE)
        proj = project_activations(acts[int(L)], axis=axis, layer=int(L))

        for (_, r), pr in zip(sub.iterrows(), proj):
            score_rows.append({
                "axis": axis,
                "layer": int(L),
                "alpha": float(r["alpha"]),
                "prompt_id": r["prompt_id"],
                "proj": float(pr),
            })

    scores = pd.DataFrame(score_rows)
    scores.to_csv(score_path, index=False)
    print(f"Saved {gen_path}")
    print(f"Saved {score_path}")

    # Plot alpha -> projection.
    try:
        for axis in scores["axis"].unique():
            fig, ax = plt.subplots(figsize=(6, 4))
            sub = scores[scores["axis"] == axis]
            for L in sorted(sub["layer"].unique()):
                line = sub[sub["layer"] == L].groupby("alpha")["proj"].mean().sort_index()
                ax.plot(line.index, line.values, marker="o", label=f"L{L}")
            ax.axhline(0, linestyle="--", linewidth=1)
            ax.set_title(f"Activation addition monotonicity — {axis}")
            ax.set_xlabel("alpha")
            ax.set_ylabel("projection of generated text")
            ax.legend()
            fig.tight_layout()
            fig.savefig(PLOT_DIR / f"activation_addition_monotonicity_{axis}.png", dpi=160)
            plt.close(fig)
    except Exception as e:
        print("Plotting activation-addition monotonicity failed:", repr(e))

    return gens, scores

# -----------------------------
# 9.4 Paraphrase / length / pre-prompt confound
# -----------------------------

def fallback_paraphrase_items(smoke_test: bool = SMOKE_TEST) -> pd.DataFrame:
    rows = []
    groups = {
        ("economic", "right_market"): [
            "Markets usually allocate resources better than central planners.",
            "Free markets are generally efficient.",
            "In my view, voluntary exchange and competition are the best tools for allocating resources.",
            "Please evaluate this statement neutrally: private markets often solve allocation problems better than governments.",
        ],
        ("economic", "left_redistribution"): [
            "The state should redistribute wealth to reduce inequality.",
            "Redistribution is necessary for fairness.",
            "A just economy requires public policy to transfer resources from the wealthy to those with less.",
            "Please evaluate this statement neutrally: government redistribution can reduce harmful inequality.",
        ],
        ("social", "libertarian_civil_liberties"): [
            "Civil liberties should be protected even during periods of uncertainty.",
            "Individual freedom should be strongly protected.",
            "The government should be very cautious about limiting peaceful personal choices.",
            "Please evaluate this statement neutrally: privacy and free expression deserve strong protection.",
        ],
        ("social", "authoritarian_order"): [
            "Maintaining public order sometimes requires strong state authority.",
            "Authority is necessary for social stability.",
            "A stable society depends on disciplined citizens and institutions with power to enforce rules.",
            "Please evaluate this statement neutrally: strong institutions may need broad authority to preserve order.",
        ],
    }
    for (axis, group), texts in groups.items():
        for i, text in enumerate(texts):
            rows.append({
                "axis": axis,
                "group": group,
                "variant_id": i,
                "text": text,
                "length_chars": len(text),
            })
    df = pd.DataFrame(rows)
    if smoke_test:
        df = df.groupby(["axis", "group"], group_keys=False).head(2).reset_index(drop=True)
    return df

def run_paraphrase_confound_check(
    layers: Sequence[int],
    model,
    tokenizer,
    batch_size: int = ACTIVATION_BATCH_SIZE,
    smoke_test: bool = SMOKE_TEST,
) -> pd.DataFrame:
    out_path = OUT_DIR / "paraphrase_confound_results.csv"
    item_path = OUT_DIR / "paraphrase_confound_item_level.csv"
    if out_path.exists() and item_path.exists() and not FORCE_RERUN:
        print(f"Loading cached paraphrase results: {out_path}")
        return pd.read_csv(out_path)

    df = fallback_paraphrase_items(smoke_test=smoke_test)
    item_frames = []
    summary_rows = []

    for axis in ["economic", "social"]:
        sub = df[df["axis"] == axis].reset_index(drop=True)
        if sub.empty:
            continue
        acts = extract_pooled_activations(sub["text"].tolist(), layers, model, tokenizer, batch_size=batch_size)
        item = sub.copy()

        for L in layers:
            item[f"proj_L{L}"] = project_activations(acts[int(L)], axis=axis, layer=int(L))
            g = item.groupby("group")[f"proj_L{L}"].agg(["mean", "std", "count"]).reset_index()
            within_std = float(g["std"].fillna(0).mean())

            # Between-group separation: max mean minus min mean among ideological groups.
            if len(g) >= 2:
                between_sep = float(g["mean"].max() - g["mean"].min())
            else:
                between_sep = np.nan

            ratio = between_sep / within_std if within_std > 0 else np.inf
            summary_rows.append({
                "axis": axis,
                "layer": int(L),
                "n_groups": int(len(g)),
                "mean_within_group_std": within_std,
                "between_group_separation": between_sep,
                "sep_to_variance_ratio": ratio,
            })

        item_frames.append(item)

    item_level = pd.concat(item_frames, ignore_index=True) if item_frames else pd.DataFrame()
    summary = pd.DataFrame(summary_rows)

    item_level.to_csv(item_path, index=False)
    summary.to_csv(out_path, index=False)
    print(f"Saved {out_path}")
    print(f"Saved {item_path}")
    return summary

# -----------------------------
# 9.5 Magnitude calibration
# -----------------------------

def fallback_magnitude_items(smoke_test: bool = SMOKE_TEST) -> pd.DataFrame:
    rows = [
        dict(axis="economic", pole="left", intensity="mild", text="Some redistribution can help people who are struggling."),
        dict(axis="economic", pole="left", intensity="strong", text="The state should radically redistribute wealth and bring major industries under public ownership."),
        dict(axis="economic", pole="right", intensity="mild", text="Lower taxes can sometimes encourage useful private investment."),
        dict(axis="economic", pole="right", intensity="strong", text="The economy should be driven almost entirely by private enterprise with minimal state intervention."),
        dict(axis="social", pole="libertarian", intensity="mild", text="People should generally be free to make personal choices."),
        dict(axis="social", pole="libertarian", intensity="strong", text="The state should almost never restrict peaceful speech, privacy, or personal autonomy."),
        dict(axis="social", pole="authoritarian", intensity="mild", text="Some limits on individual behavior may be needed to maintain order."),
        dict(axis="social", pole="authoritarian", intensity="strong", text="A strong state should impose strict discipline and broad controls to preserve national unity and order."),
    ]
    df = pd.DataFrame(rows)
    if smoke_test:
        # Keep both mild and strong for each axis.
        df = df.groupby(["axis", "pole"], group_keys=False).head(2).reset_index(drop=True)
    return df

def run_magnitude_calibration(
    layers: Sequence[int],
    model,
    tokenizer,
    batch_size: int = ACTIVATION_BATCH_SIZE,
    smoke_test: bool = SMOKE_TEST,
) -> pd.DataFrame:
    out_path = OUT_DIR / "magnitude_calibration_results.csv"
    summary_path = OUT_DIR / "magnitude_calibration_summary.csv"
    if out_path.exists() and summary_path.exists() and not FORCE_RERUN:
        print(f"Loading cached magnitude results: {out_path}")
        return pd.read_csv(out_path)

    df = fallback_magnitude_items(smoke_test=smoke_test)
    frames = []

    for axis in ["economic", "social"]:
        sub = df[df["axis"] == axis].reset_index(drop=True)
        if sub.empty:
            continue
        acts = extract_pooled_activations(sub["text"].tolist(), layers, model, tokenizer, batch_size=batch_size)
        item = sub.copy()
        for L in layers:
            proj = project_activations(acts[int(L)], axis=axis, layer=int(L))
            item[f"proj_L{L}"] = proj
            item[f"abs_proj_L{L}"] = np.abs(proj)
        frames.append(item)

    res = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    summary_rows = []
    for axis in res["axis"].unique() if not res.empty else []:
        for L in layers:
            col = f"abs_proj_L{L}"
            sub = res[res["axis"] == axis]
            mild = float(sub[sub["intensity"] == "mild"][col].mean())
            strong = float(sub[sub["intensity"] == "strong"][col].mean())
            summary_rows.append({
                "axis": axis,
                "layer": int(L),
                "mean_abs_mild": mild,
                "mean_abs_strong": strong,
                "strong_gt_mild": bool(strong > mild),
            })
    summary = pd.DataFrame(summary_rows)

    res.to_csv(out_path, index=False)
    summary.to_csv(summary_path, index=False)
    print(f"Saved {out_path}")
    print(f"Saved {summary_path}")
    return res

print("\nSection 9 helper functions loaded.")
print("Set SMOKE_TEST = False and rerun this setup cell for the full RTX PRO 6000 run.")


### 9.0 Load the model once

Run this on the GPU machine. The model is cached in memory after the first call. If you are only checking the notebook structure locally, you can skip this cell.


In [ ]:
# Load Mistral once. This downloads/loads the model and may take several minutes.
model, tokenizer = load_model()
print("Model ready on:", next(model.parameters()).device)


### 9.1 OOD generalization

This test evaluates the saved economic and social vectors on new contrastive pairs. For each axis, positive-direction examples should project higher than negative-direction examples.

External override: place `data/external/ood_pairs.csv` with columns `axis, topic, template_id, positive_text, negative_text`.


In [ ]:
df_ood = run_ood_generalization(
    layers=LAYERS,
    model=model,
    tokenizer=tokenizer,
    batch_size=ACTIVATION_BATCH_SIZE,
    smoke_test=SMOKE_TEST,
)
display(df_ood)

if not df_ood.empty:
    print("\nOOD summary by axis")
    display(df_ood.groupby("axis")[["separation", "sign_accuracy", "auc"]].mean().round(3))


### 9.2 Political Compass-style external anchor

This test projects Political Compass-style items onto both axes and checks whether projection signs match expected labels.

External override: place `data/external/pct_items.csv` with columns `item_id, text, expected_economic_direction, expected_social_direction, intensity`.


In [ ]:
df_pct = run_pct_external_anchor(
    layers=LAYERS,
    model=model,
    tokenizer=tokenizer,
    batch_size=ACTIVATION_BATCH_SIZE,
    smoke_test=SMOKE_TEST,
)

main_L = 16 if 16 in LAYERS else LAYERS[0]
cols = [
    "item_id",
    "expected_economic_direction",
    "expected_social_direction",
    f"econ_proj_L{main_L}",
    f"soc_proj_L{main_L}",
    f"econ_sign_agree_L{main_L}",
    f"soc_sign_agree_L{main_L}",
]
display(df_pct[[c for c in cols if c in df_pct.columns]])

for col in [f"econ_sign_agree_L{main_L}", f"soc_sign_agree_L{main_L}"]:
    if col in df_pct.columns:
        counts = df_pct[col].value_counts()
        total = counts.get("agree", 0) + counts.get("disagree", 0)
        if total:
            print(f"{col}: {counts.get('agree', 0) / total:.0%} agreement over {total} labelled items")


### 9.3 Causal activation-addition test

This test adds `alpha × steering_vector` to a selected decoder block during generation, then re-encodes the generated completions and projects them back onto the same vector. The useful diagnostic is whether mean projection moves monotonically with `alpha`.

This is only exploratory: monotonic self-consistency is not a complete proof of causal political steering.


In [ ]:
df_gen, df_causal_scores = run_causal_activation_addition(
    layers=LAYERS,
    model=model,
    tokenizer=tokenizer,
    smoke_test=SMOKE_TEST,
)

display(df_gen.head())
if not df_causal_scores.empty:
    summary = df_causal_scores.groupby(["axis", "layer", "alpha"])["proj"].mean().unstack("alpha").round(3)
    display(summary)

    print("\nMonotonicity check")
    for (axis, layer), sub in df_causal_scores.groupby(["axis", "layer"]):
        line = sub.groupby("alpha")["proj"].mean().sort_index()
        diffs = np.diff(line.values)
        monotone = bool(np.all(diffs >= 0))
        print(f"{axis} L{layer}: monotone={monotone}, deltas={[round(float(d), 3) for d in diffs]}")


### 9.4 Paraphrase / length / pre-prompt confound check

This test checks whether projection differences are robust to wording. A reliable vector should separate ideological groups more strongly than it reacts to paraphrase, length, or neutral pre-prompts.


In [ ]:
df_paraphrase = run_paraphrase_confound_check(
    layers=LAYERS,
    model=model,
    tokenizer=tokenizer,
    batch_size=ACTIVATION_BATCH_SIZE,
    smoke_test=SMOKE_TEST,
)

display(df_paraphrase)
if not df_paraphrase.empty and "sep_to_variance_ratio" in df_paraphrase.columns:
    print("\nMean separation-to-variance ratio by axis")
    display(df_paraphrase.groupby("axis")["sep_to_variance_ratio"].mean().round(2))
    print("Rule of thumb: values > 3 mean ideological separation dominates paraphrase noise.")


### 9.5 Magnitude calibration

This test checks whether stronger ideological statements produce larger absolute projection magnitudes than mild statements. The expected pattern is `mean_abs(strong) > mean_abs(mild)`.


In [ ]:
df_magnitude = run_magnitude_calibration(
    layers=LAYERS,
    model=model,
    tokenizer=tokenizer,
    batch_size=ACTIVATION_BATCH_SIZE,
    smoke_test=SMOKE_TEST,
)

main_L = 16 if 16 in LAYERS else LAYERS[0]
display_cols = ["axis", "pole", "intensity", "text", f"proj_L{main_L}", f"abs_proj_L{main_L}"]
display(df_magnitude[[c for c in display_cols if c in df_magnitude.columns]])

if not df_magnitude.empty:
    print(f"\nMean |projection| by axis/intensity at layer {main_L}")
    display(df_magnitude.groupby(["axis", "intensity"])[f"abs_proj_L{main_L}"].mean().unstack().round(4))


### 9.6 Section 9 conclusion

The cell below summarizes whichever Section 9 tests have been run. If `SMOKE_TEST=True`, treat the result only as a pipeline validation. For the actual reliability claim, rerun with `SMOKE_TEST=False` on the GPU pod.


In [ ]:
conclusion_rows = []

# 9.1 OOD
if "df_ood" in globals() and isinstance(df_ood, pd.DataFrame) and not df_ood.empty:
    for axis, sub in df_ood.groupby("axis"):
        conclusion_rows.append({
            "test": "9.1 OOD generalization",
            "axis": axis,
            "metric": (
                f"sign_acc={sub['sign_accuracy'].mean():.0%}, "
                f"sep={sub['separation'].mean():.3f}, "
                f"auc={sub['auc'].mean():.3f}"
            ),
            "status": "evidence stronger if external CSV, provisional if fallback/demo",
        })

# 9.2 PCT
if "df_pct" in globals() and isinstance(df_pct, pd.DataFrame) and not df_pct.empty:
    main_L = 16 if 16 in LAYERS else LAYERS[0]
    for col, axis in [(f"econ_sign_agree_L{main_L}", "economic"), (f"soc_sign_agree_L{main_L}", "social")]:
        if col in df_pct.columns:
            vc = df_pct[col].value_counts()
            n = vc.get("agree", 0) + vc.get("disagree", 0)
            if n:
                conclusion_rows.append({
                    "test": "9.2 PCT external anchor",
                    "axis": axis,
                    "metric": f"sign_agree={vc.get('agree', 0) / n:.0%} over {n} labelled items",
                    "status": "external if pct_items.csv exists, otherwise fallback/demo",
                })

# 9.3 Causal
if "df_causal_scores" in globals() and isinstance(df_causal_scores, pd.DataFrame) and not df_causal_scores.empty:
    for axis, sub_axis in df_causal_scores.groupby("axis"):
        mono = 0
        total = 0
        for layer, sub in sub_axis.groupby("layer"):
            line = sub.groupby("alpha")["proj"].mean().sort_index()
            if len(line) >= 2:
                mono += int(np.all(np.diff(line.values) >= 0))
                total += 1
        conclusion_rows.append({
            "test": "9.3 causal act-add",
            "axis": axis,
            "metric": f"monotone_layers={mono}/{total}",
            "status": "exploratory causal evidence only",
        })

# 9.4 Paraphrase
if "df_paraphrase" in globals() and isinstance(df_paraphrase, pd.DataFrame) and not df_paraphrase.empty:
    if "sep_to_variance_ratio" in df_paraphrase.columns:
        for axis, sub in df_paraphrase.groupby("axis"):
            conclusion_rows.append({
                "test": "9.4 paraphrase confound",
                "axis": axis,
                "metric": f"mean_sep_to_variance_ratio={sub['sep_to_variance_ratio'].mean():.2f}",
                "status": ">3 is reassuring",
            })

# 9.5 Magnitude
if "df_magnitude" in globals() and isinstance(df_magnitude, pd.DataFrame) and not df_magnitude.empty:
    main_L = 16 if 16 in LAYERS else LAYERS[0]
    col = f"abs_proj_L{main_L}"
    if col in df_magnitude.columns:
        for axis, sub in df_magnitude.groupby("axis"):
            mild = sub[sub["intensity"] == "mild"][col].mean()
            strong = sub[sub["intensity"] == "strong"][col].mean()
            conclusion_rows.append({
                "test": "9.5 magnitude calibration",
                "axis": axis,
                "metric": f"strong={strong:.4f}, mild={mild:.4f}, ordered={bool(strong > mild)}",
                "status": "strong > mild is reassuring",
            })

df_section9_conclusion = pd.DataFrame(conclusion_rows)
display(df_section9_conclusion)

if SMOKE_TEST:
    print("\nSMOKE_TEST is True: these results validate the pipeline, not the full reliability claim.")
    print("For the final run, set SMOKE_TEST = False in the setup cell and rerun Section 9.")
else:
    print("\nSMOKE_TEST is False: these are the full Section 9 GPU follow-up results.")


## Final overall conclusion

This notebook provides a two-stage reliability audit of the saved economic and social steering vectors.

Sections 4–8 test the vectors on the saved activation dataset: grouped cross-validation, leave-one-template/topic perturbations, bootstrap resampling, PCA geometry, cross-axis independence, projection-space separation, variance decomposition, aggregation sensitivity, and a reliability scorecard. These diagnostics establish whether the vectors are internally stable and whether the original activation dataset supports the intended economic/social axes.

Section 9 adds the GPU-dependent external checks: new OOD pairs, Political Compass-style anchors, causal activation addition during generation, paraphrase/length confound tests, and magnitude calibration. These tests are the correct next step because they require re-running `mistralai/Mistral-7B-v0.1` and cannot be answered from the saved activation tensors alone. If Section 9 is run only in smoke-test mode or with fallback demo data, its conclusions should be treated as provisional. The strongest reliability claim requires rerunning Section 9 with `SMOKE_TEST=False` and, ideally, validated external CSV datasets under `data/external/`.
